# 02 — Data Download Verification
**Project:** Real-Time Audio Fall Detection (TinyML)

This notebook systematically verifies that all datasets downloaded by
`01_data_download.ipynb` are present and intact on Google Drive.

### Checks performed
| Check | Description |
|-------|-------------|
| Exists | Folder exists and is non-empty |
| File count | Number of files matches expected minimum |
| Size | Total size is within expected range |
| Audio integrity | A sample of audio files can be opened and decoded |
| Empty / truncated files | Detect 0-byte or suspiciously small files |
| Expected sub-structure | Key sub-folders or files are present |

## 1 · Setup

In [ ]:
from google.colab import drive
import os, random, subprocess
from pathlib import Path
from collections import Counter

drive.mount("/content/drive")

# Install soundfile for audio validation
subprocess.run(["pip", "install", "-q", "soundfile"], check=True)
import soundfile as sf

# ── Project paths (must match 01_data_download) ──
BASE_DIR = "/content/drive/MyDrive/DL-Project"
RAW_DIR  = os.path.join(BASE_DIR, "raw")

DATASETS = ["DESED", "CAUCAFall", "WaterLeakage", "HumanScreaming", "FSD50K", "SAFE", "AudioSet"]

print(f"Raw data directory: {RAW_DIR}")
print(f"Exists: {os.path.isdir(RAW_DIR)}")

In [ ]:
# ── Helper functions ───────────────────────────────────────────

def count_files(directory):
    """Recursively count files."""
    return sum(1 for _ in Path(directory).rglob("*") if _.is_file())


def dir_size_mb(directory):
    """Total size in MB."""
    return sum(f.stat().st_size for f in Path(directory).rglob("*") if f.is_file()) / (1024 * 1024)


def get_extension_counts(directory):
    """Count files by extension."""
    exts = Counter()
    for f in Path(directory).rglob("*"):
        if f.is_file():
            exts[f.suffix.lower()] += 1
    return exts


def find_empty_or_tiny_files(directory, min_bytes=100):
    """Find files that are 0-byte or smaller than min_bytes."""
    bad = []
    for f in Path(directory).rglob("*"):
        if f.is_file() and f.stat().st_size < min_bytes:
            bad.append((str(f), f.stat().st_size))
    return bad


def find_leftover_archives(directory):
    """Find .zip / .tar.gz files that may not have been extracted."""
    archives = []
    for ext in ("*.zip", "*.tar.gz", "*.tgz", "*.tar"):
        archives.extend(Path(directory).rglob(ext))
    return [str(a) for a in archives]


AUDIO_EXTS = {".wav", ".flac", ".ogg", ".mp3", ".aac", ".m4a", ".wma", ".aiff"}

def collect_audio_files(directory):
    """Collect all audio file paths."""
    return [f for f in Path(directory).rglob("*")
            if f.is_file() and f.suffix.lower() in AUDIO_EXTS]


def sample_audio_check(directory, n_samples=20):
    """Try to open a random sample of audio files with soundfile.
    Returns (n_ok, n_fail, failures_list).
    """
    audio_files = collect_audio_files(directory)
    if not audio_files:
        return 0, 0, []

    sample = random.sample(audio_files, min(n_samples, len(audio_files)))
    ok, fail, failures = 0, 0, []

    for fp in sample:
        try:
            info = sf.info(str(fp))
            if info.frames == 0:
                fail += 1
                failures.append((str(fp), "0 frames"))
            else:
                ok += 1
        except Exception as e:
            fail += 1
            failures.append((str(fp), str(e)[:80]))

    return ok, fail, failures


print("Verification helpers loaded.")

## 2 · Expected dataset profiles

These are approximate expected values. Adjust if you know the exact numbers from your download run.

In [ ]:
# ── Expected profiles (approximate) ────────────────────────────
# min_files : hard lower-bound — below this something is clearly wrong
# min_mb    : rough minimum total size in MB
# audio_exts: expected dominant audio extensions
# key_items : sub-folders or files that should be present

EXPECTED = {
    "DESED": {
        "min_files": 10,
        "min_mb": 50,
        "audio_exts": {".wav"},
        "key_items": [],   # Zenodo structure varies
        "source": "Zenodo — https://zenodo.org/records/4560938",
    },
    "CAUCAFall": {
        "min_files": 10,
        "min_mb": 5,
        "audio_exts": {".wav", ".mp3"},
        "key_items": [],
        "source": "Mendeley — https://data.mendeley.com/datasets/7w7fccy7ky/4",
    },
    "WaterLeakage": {
        "min_files": 5,
        "min_mb": 1,
        "audio_exts": {".wav", ".mp3"},
        "key_items": [],
        "source": "GitHub — aliEmreOzturk/water_leakage_voice_data",
    },
    "HumanScreaming": {
        "min_files": 100,
        "min_mb": 50,
        "audio_exts": {".wav", ".mp3", ".flac"},
        "key_items": [],
        "source": "Kaggle — whats2000/human-screaming-detection-dataset",
    },
    "FSD50K": {
        "min_files": 1000,
        "min_mb": 500,
        "audio_exts": {".wav", ".flac"},
        "key_items": [],
        "source": "Zenodo — https://zenodo.org/records/4060432",
    },
    "SAFE": {
        "min_files": 10,
        "min_mb": 5,
        "audio_exts": {".wav", ".mp3"},
        "key_items": [],
        "source": "Kaggle — antonygarciag/fall-audio-detection-dataset",
    },
    "AudioSet": {
        "min_files": 10,
        "min_mb": 1,
        "audio_exts": {".wav"},
        "key_items": [],
        "source": "YouTube via yt-dlp",
    },
}

print(f"Verification profiles defined for {len(EXPECTED)} datasets.")

## 3 · Run full verification

In [ ]:
# ── Full verification pass ─────────────────────────────────────

PASS = "\033[92mPASS\033[0m"
WARN = "\033[93mWARN\033[0m"
FAIL = "\033[91mFAIL\033[0m"

results = {}  # dataset -> {status, issues}

for ds in DATASETS:
    ds_path = os.path.join(RAW_DIR, ds)
    exp = EXPECTED[ds]
    issues = []

    print("=" * 65)
    print(f"  {ds}")
    print(f"  Source: {exp['source']}")
    print("-" * 65)

    # ── Check 1: Folder exists ──
    if not os.path.isdir(ds_path):
        print(f"  {FAIL}  Folder does not exist: {ds_path}")
        issues.append("MISSING folder")
        results[ds] = {"status": "FAIL", "issues": issues}
        print()
        continue

    # ── Check 2: File count ──
    n_files = count_files(ds_path)
    if n_files == 0:
        print(f"  {FAIL}  Folder is EMPTY")
        issues.append("EMPTY folder")
        results[ds] = {"status": "FAIL", "issues": issues}
        print()
        continue
    elif n_files < exp["min_files"]:
        print(f"  {WARN}  File count: {n_files}  (expected >= {exp['min_files']})")
        issues.append(f"Low file count: {n_files} < {exp['min_files']}")
    else:
        print(f"  {PASS}  File count: {n_files}  (expected >= {exp['min_files']})")

    # ── Check 3: Total size ──
    size = dir_size_mb(ds_path)
    if size < exp["min_mb"]:
        print(f"  {WARN}  Total size: {size:.1f} MB  (expected >= {exp['min_mb']} MB)")
        issues.append(f"Small size: {size:.1f} MB < {exp['min_mb']} MB")
    else:
        print(f"  {PASS}  Total size: {size:.1f} MB  (expected >= {exp['min_mb']} MB)")

    # ── Check 4: Extension distribution ──
    ext_counts = get_extension_counts(ds_path)
    print(f"  INFO  Extensions: ", end="")
    for ext, cnt in ext_counts.most_common(8):
        print(f"{ext}={cnt}  ", end="")
    print()

    audio_files = collect_audio_files(ds_path)
    if len(audio_files) == 0:
        print(f"  {WARN}  No audio files found (expected {exp['audio_exts']})")
        issues.append("No audio files found")
    else:
        print(f"  {PASS}  Audio files: {len(audio_files)}")

    # ── Check 5: Empty / tiny files ──
    tiny = find_empty_or_tiny_files(ds_path)
    if tiny:
        print(f"  {WARN}  Found {len(tiny)} empty / tiny file(s) (<100 bytes):")
        for fpath, sz in tiny[:5]:
            print(f"         {os.path.basename(fpath)}  ({sz} bytes)")
        if len(tiny) > 5:
            print(f"         ... and {len(tiny) - 5} more")
        issues.append(f"{len(tiny)} empty/tiny files")
    else:
        print(f"  {PASS}  No empty/tiny files")

    # ── Check 6: Leftover archives ──
    archives = find_leftover_archives(ds_path)
    if archives:
        print(f"  {WARN}  Leftover archives (may not have been extracted):")
        for a in archives[:5]:
            print(f"         {os.path.basename(a)}")
        issues.append(f"{len(archives)} leftover archive(s)")
    else:
        print(f"  {PASS}  No leftover archives")

    # ── Check 7: Audio integrity (sample) ──
    if audio_files:
        n_ok, n_fail, failures = sample_audio_check(ds_path, n_samples=20)
        total_checked = n_ok + n_fail
        if n_fail > 0:
            print(f"  {WARN}  Audio sample check: {n_ok}/{total_checked} OK, {n_fail} FAILED")
            for fpath, err in failures[:3]:
                print(f"         {os.path.basename(fpath)}: {err}")
            issues.append(f"{n_fail}/{total_checked} audio samples unreadable")
        else:
            print(f"  {PASS}  Audio sample check: {n_ok}/{total_checked} OK")

    # ── Check 8: Key items ──
    for item in exp.get("key_items", []):
        item_path = os.path.join(ds_path, item)
        if os.path.exists(item_path):
            print(f"  {PASS}  Key item present: {item}")
        else:
            print(f"  {WARN}  Key item MISSING: {item}")
            issues.append(f"Missing key item: {item}")

    # ── Verdict ──
    if not issues:
        results[ds] = {"status": "PASS", "issues": []}
    elif any("EMPTY" in i or "MISSING" in i for i in issues):
        results[ds] = {"status": "FAIL", "issues": issues}
    else:
        results[ds] = {"status": "WARN", "issues": issues}

    print()

## 4 · Summary report

In [ ]:
# ── Final summary ──────────────────────────────────────────────

STATUS_ICON = {"PASS": "\u2705", "WARN": "\u26A0\uFE0F", "FAIL": "\u274C"}

print("=" * 65)
print("  VERIFICATION SUMMARY")
print("=" * 65)

all_ok = True
for ds in DATASETS:
    r = results.get(ds, {"status": "FAIL", "issues": ["Not checked"]})
    icon = STATUS_ICON.get(r["status"], "?")
    line = f"  {icon}  {ds:20s}  {r['status']}"
    if r["issues"]:
        line += f"  — {'; '.join(r['issues'])}"
        all_ok = False
    print(line)

print("=" * 65)

if all_ok:
    print("\n  All datasets verified successfully!")
    print("  You can proceed to the preprocessing step.")
else:
    print("\n  Some datasets have issues. Review the warnings/failures above.")
    print("  Re-run the corresponding cell in 01_data_download.ipynb to fix.")

## 5 · Detailed directory tree (optional)

Run the cell below to print the top-level structure of each dataset folder.

In [ ]:
# ── Directory tree (depth=2) ────────────────────────────────────

for ds in DATASETS:
    ds_path = os.path.join(RAW_DIR, ds)
    print(f"\n{'=' * 50}")
    print(f"  {ds}/")
    print(f"{'=' * 50}")

    if not os.path.isdir(ds_path):
        print("  (does not exist)")
        continue

    ds_root = Path(ds_path)
    # Show top-level items
    items = sorted(ds_root.iterdir())
    for item in items:
        if item.is_dir():
            sub_count = count_files(item)
            print(f"  {item.name}/  ({sub_count} files)")
            # Show second level
            for sub in sorted(item.iterdir())[:10]:
                if sub.is_dir():
                    print(f"    {sub.name}/  ({count_files(sub)} files)")
                else:
                    print(f"    {sub.name}  ({sub.stat().st_size / 1024:.0f} KB)")
            sub_items = list(item.iterdir())
            if len(sub_items) > 10:
                print(f"    ... and {len(sub_items) - 10} more items")
        else:
            print(f"  {item.name}  ({item.stat().st_size / 1024:.0f} KB)")

print("\nDone.")